# Working without Momentum Decay

In [3]:
import numpy as np
from tqdm import tqdm

# Stats for both players
# P1Name = "Alcaraz"
# P2Name = "Zverev"

# P1 = {
#     'first_in':      0.64,   
#     'win_first':     0.76,
#     'win_second':    0.57,
#     'return_first':  0.35,
#     'return_second': 0.55,
# }

# P2 = {
#     'first_in':      0.73,
#     'win_first':     0.74,
#     'win_second':    0.53,
#     'return_first':  0.28,   
#     'return_second': 0.51,   
# }

P1Name = "Fonseca"
P2Name = "Novak"
P1 = {
    'first_in':      0.74,   
    'win_first':     0.69,
    'win_second':    0.43,
    'return_first':  0.32,
    'return_second': 0.47,
}

P2 = {
    'first_in':      0.71,
    'win_first':     0.68,
    'win_second':    0.53,
    'return_first':  0.31,   
    'return_second': 0.57,   
}

def sim_point(server_first_in,
              server_win_first, server_win_second,
              receiver_return_first, receiver_return_second):
    """
    server_win_first      : server's historical 1st serve win %
    receiver_return_first : receiver's historical return win % on 1st serve
    """
    # Blend server and receiver perspectives
    p_win_1st = (server_win_first + (1 - receiver_return_first)) / 2
    p_win_2nd = (server_win_second + (1 - receiver_return_second)) / 2

    if np.random.random() < server_first_in:
        return np.random.random() < p_win_1st
    else:
        return np.random.random() < p_win_2nd

def sim_game(p1_serving):
    score = [0, 0]
    
    while True:
        if p1_serving:
            # P1 serving, P2 receiving
            p1_won = sim_point(
                server_first_in      = P1['first_in'],
                server_win_first     = P1['win_first'],
                server_win_second    = P1['win_second'],
                receiver_return_first  = P2['return_first'],
                receiver_return_second = P2['return_second'],
            )
        else:
            # P2 serving, P1 receiving
            p2_won = sim_point(
                server_first_in      = P2['first_in'],
                server_win_first     = P2['win_first'],
                server_win_second    = P2['win_second'],
                receiver_return_first  = P1['return_first'],
                receiver_return_second = P1['return_second'],
            )
            p1_won = not p2_won

        if p1_won: score[0] += 1
        else:      score[1] += 1

        if score[0] >= 4 and score[0] - score[1] >= 2:
            return True
        if score[1] >= 4 and score[1] - score[0] >= 2:
            return False

def sim_tiebreak(p1_serving):
    """First to 7, win by 2. Server alternates every 2 points."""
    score = [0, 0]
    point_count = 0
    
    while True:
        # Server switches every 2 points after first point
        if point_count == 0:
            p1_serves_now = p1_serving
        else:
            p1_serves_now = (p1_serving) == (point_count % 2 == 0)
        
        if p1_serves_now:
            p1_won = sim_point(
                server_first_in      = P1['first_in'],
                server_win_first     = P1['win_first'],
                server_win_second    = P1['win_second'],
                receiver_return_first  = P2['return_first'],
                receiver_return_second = P2['return_second'],
            )
        else:
            p1_won = not sim_point(
                server_first_in      = P2['first_in'],
                server_win_first     = P2['win_first'],
                server_win_second    = P2['win_second'],
                receiver_return_first  = P1['return_first'],
                receiver_return_second = P1['return_second'],
            )
        
        if p1_won: score[0] += 1
        else:      score[1] += 1
        point_count += 1
        
        if score[0] >= 7 and score[0] - score[1] >= 2:
            return True
        if score[1] >= 7 and score[1] - score[0] >= 2:
            return False

def sim_set(p1_serving):
    """Returns (True if P1 wins set, who serves next set)."""
    games = [0, 0]
    
    while True:
        p1_won_game = sim_game(p1_serving)
        
        if p1_won_game: games[0] += 1
        else:           games[1] += 1
        
        p1_serving = not p1_serving  # server alternates each game
        
        # Tiebreak at 6-6
        if games[0] == 6 and games[1] == 6:
            p1_won_tb = sim_tiebreak(p1_serving)
            if p1_won_tb: games[0] += 1
            else:          games[1] += 1
            return (games[0] > games[1]), p1_serving
        
        # Normal set win: 6+ games, win by 2
        if games[0] >= 6 and games[0] - games[1] >= 2:
            return True, p1_serving
        if games[1] >= 6 and games[1] - games[0] >= 2:
            return False, p1_serving

def sim_match(p1_serves_first=True, best_of=3):
    """Returns True if P1 wins the match."""
    sets_needed = best_of // 2 + 1
    sets = [0, 0]
    p1_serving = p1_serves_first
    
    while sets[0] < sets_needed and sets[1] < sets_needed:
        p1_won_set, p1_serving = sim_set(p1_serving)
        if p1_won_set: sets[0] += 1
        else:          sets[1] += 1
    
    return sets[0] > sets[1]

def run_simulation(N=100_000):
    wins = sum(sim_match(p1_serves_first=True, best_of=5) for _ in tqdm(range(N)))
    return wins / N

p1_win_prob = run_simulation(N=100_000)
print(f"{P1Name} win probability: {p1_win_prob:.4f}")
print(f"{P2Name} win probability: {1 - p1_win_prob:.4f}")

100%|██████████| 100000/100000 [00:20<00:00, 4877.97it/s]

Fonseca win probability: 0.4111
Novak win probability: 0.5889


# Momentum Decay

In [2]:
import numpy as np

# ─────────────────────────────────────────
#  BETA MODEL (serve or return)
# ─────────────────────────────────────────

class BetaModel:
    """
    Tracks a single probability (serve win rate OR return win rate)
    using a Beta distribution with exponential decay on in-match evidence.

    prior_strength : number of virtual career points anchoring the prior
    lam            : decay factor applied to in-match counts each point (0.90)
    warmup         : number of points before in-match component is used
                     defaults to effective window = 1 / (1 - lam)
    """

    def __init__(self, career_rate, prior_strength=200, lam=0.90, warmup=None):
        # Fixed prior — never decayed, never updated
        self.alpha_prior = career_rate * prior_strength
        self.beta_prior  = (1 - career_rate) * prior_strength

        # In-match component — decayed each point
        self.alpha_match = 0.0
        self.beta_match  = 0.0

        self.lam     = lam
        self.warmup  = warmup if warmup is not None else int(1 / (1 - lam))
        self.n_obs   = 0   # count of observed points so far

    def get_p(self):
        """Return current probability estimate."""
        if self.n_obs < self.warmup:
            # Pure career stats during warmup
            return self.alpha_prior / (self.alpha_prior + self.beta_prior)

        total = (self.alpha_prior + self.beta_prior +
                 self.alpha_match + self.beta_match)
        return (self.alpha_prior + self.alpha_match) / total

    def update(self, won):
        """
        Call after every point.
        won = True  if the player this model tracks won the point
        won = False otherwise
        """
        # Step 1: decay in-match component
        self.alpha_match *= self.lam
        self.beta_match  *= self.lam

        # Step 2: add new observation
        if won:
            self.alpha_match += 1.0
        else:
            self.beta_match  += 1.0

        self.n_obs += 1

    def reset(self):
        """Reset in-match state (call between simulations)."""
        self.alpha_match = 0.0
        self.beta_match  = 0.0
        self.n_obs       = 0


# ─────────────────────────────────────────
#  PLAYER
# ─────────────────────────────────────────

class Player:
    """
    Holds all stats and Beta models for one player.

    Stats expected:
      first_in        : 1st serve in %
      win_first       : win % on 1st serve points (serving)
      win_second      : win % on 2nd serve points (serving)
      return_first    : win % returning 1st serve
      return_second   : win % returning 2nd serve
    """

    def __init__(self, name, stats, prior_strength=200, lam=0.95):
        self.name  = name
        self.stats = stats

        # Serve models (one per serve type)
        self.serve_first_model  = BetaModel(stats['win_first'],
                                            prior_strength, lam)
        self.serve_second_model = BetaModel(stats['win_second'],
                                            prior_strength, lam)

        # Return models (one per serve type received)
        self.return_first_model  = BetaModel(stats['return_first'],
                                             prior_strength, lam)
        self.return_second_model = BetaModel(stats['return_second'],
                                             prior_strength, lam)

    def reset(self):
        """Reset all in-match state between simulations."""
        self.serve_first_model.reset()
        self.serve_second_model.reset()
        self.return_first_model.reset()
        self.return_second_model.reset()


# ─────────────────────────────────────────
#  POINT SIMULATION
# ─────────────────────────────────────────

def sim_point(server: Player, receiver: Player):
    """
    Simulate one point. Returns True if server wins.
    Updates all four relevant Beta models after the point.
    """
    first_in = server.stats['first_in']

    # Get current probability estimates
    p_serve_1st  = server.serve_first_model.get_p()
    p_serve_2nd  = server.serve_second_model.get_p()
    p_return_1st = receiver.return_first_model.get_p()
    p_return_2nd = receiver.return_second_model.get_p()

    # Blend server and receiver perspectives
    p_win_1st = (p_serve_1st + (1 - p_return_1st)) / 2
    p_win_2nd = (p_serve_2nd + (1 - p_return_2nd)) / 2

    # Simulate serve and point outcome
    if np.random.random() < first_in:
        # 1st serve in
        server_won = np.random.random() < p_win_1st
        # Update serve models (1st serve)
        server.serve_first_model.update(server_won)
        receiver.return_first_model.update(not server_won)
    else:
        # 2nd serve
        server_won = np.random.random() < p_win_2nd
        # Update serve models (2nd serve)
        server.serve_second_model.update(server_won)
        receiver.return_second_model.update(not server_won)

    return server_won


# ─────────────────────────────────────────
#  GAME SIMULATION
# ─────────────────────────────────────────

def sim_game(server: Player, receiver: Player):
    """
    Simulate one game. Returns True if server wins.
    Server/receiver identity stays fixed for the whole game.
    """
    # Tennis point score: 0,1,2,3 = 0,15,30,40
    score = [0, 0]   # [server, receiver]

    while True:
        server_won = sim_point(server, receiver)

        if server_won:
            score[0] += 1
        else:
            score[1] += 1

        # Win conditions: reach 4+ points AND lead by 2+
        if score[0] >= 4 and score[0] - score[1] >= 2:
            return True
        if score[1] >= 4 and score[1] - score[0] >= 2:
            return False


# ─────────────────────────────────────────
#  TIEBREAK SIMULATION
# ─────────────────────────────────────────

def sim_tiebreak(p1: Player, p2: Player, p1_serves_first: bool):
    """
    Simulate a tiebreak. First to 7, win by 2.
    Server alternates: 1 point, then every 2 points.
    Returns True if p1 wins.
    """
    score = [0, 0]
    point_count = 0

    while True:
        # Determine who serves this point
        # First point: initial server. Then alternates every 2
        if point_count == 0:
            p1_serves = p1_serves_first
        else:
            # After point 0: server flips every 2 points
            p1_serves = p1_serves_first == (point_count % 2 == 0)

        server   = p1 if p1_serves else p2
        receiver = p2 if p1_serves else p1

        server_won = sim_point(server, receiver)
        p1_won = server_won if p1_serves else not server_won

        if p1_won:
            score[0] += 1
        else:
            score[1] += 1

        point_count += 1

        if score[0] >= 7 and score[0] - score[1] >= 2:
            return True
        if score[1] >= 7 and score[1] - score[0] >= 2:
            return False


# ─────────────────────────────────────────
#  SET SIMULATION
# ─────────────────────────────────────────

def sim_set(p1: Player, p2: Player, p1_serves_first: bool):
    """
    Simulate one set. Returns (True if p1 wins, who serves next set).
    """
    games = [0, 0]
    p1_serving = p1_serves_first

    while True:
        server   = p1 if p1_serving else p2
        receiver = p2 if p1_serving else p1

        server_won = sim_game(server, receiver)
        p1_won_game = server_won if p1_serving else not server_won

        if p1_won_game:
            games[0] += 1
        else:
            games[1] += 1

        # Server alternates each game
        p1_serving = not p1_serving

        # Tiebreak at 6-6
        if games[0] == 6 and games[1] == 6:
            p1_won_tb = sim_tiebreak(p1, p2, p1_serving)
            if p1_won_tb:
                games[0] += 1
            else:
                games[1] += 1
            return (games[0] > games[1]), p1_serving

        # Normal set win
        if games[0] >= 6 and games[0] - games[1] >= 2:
            return True, p1_serving
        if games[1] >= 6 and games[1] - games[0] >= 2:
            return False, p1_serving


# ─────────────────────────────────────────
#  MATCH SIMULATION
# ─────────────────────────────────────────

def sim_match(p1: Player, p2: Player,
              p1_serves_first: bool = True,
              best_of: int = 3):
    """
    Simulate one full match. Returns True if p1 wins.
    Resets both players' in-match state at the start.
    """
    p1.reset()
    p2.reset()

    sets_needed = best_of // 2 + 1
    sets = [0, 0]
    p1_serving = p1_serves_first

    while sets[0] < sets_needed and sets[1] < sets_needed:
        p1_won_set, p1_serving = sim_set(p1, p2, p1_serving)
        if p1_won_set:
            sets[0] += 1
        else:
            sets[1] += 1

    return sets[0] > sets[1]


# ─────────────────────────────────────────
#  RUN SIMULATION
# ─────────────────────────────────────────

def run_simulation(p1: Player, p2: Player,
                   N: int = 100_000,
                   p1_serves_first: bool = True,
                   best_of: int = 3):
    """
    Run N simulations. Returns P1 win probability.
    """
    wins = 0
    for _ in range(N):
        if sim_match(p1, p2, p1_serves_first, best_of):
            wins += 1
    return wins / N


# ─────────────────────────────────────────
#  EXAMPLE USAGE
# ─────────────────────────────────────────

if __name__ == "__main__":

    P1Name = "Sinner"
    P2Name = "Djokovic"
    p1_stats= {
        'first_in':      0.62,   
        'win_first':     0.81,
        'win_second':    0.59,
        'return_first':  0.33,
        'return_second': 0.59,
    }

    p2_stats = {
        'first_in':      0.66,
        'win_first':     0.77,
        'win_second':    0.57,
        'return_first':  0.31,   
        'return_second': 0.51,   
    }

    p1 = Player("P1", p1_stats, prior_strength=100, lam=0.90)
    p2 = Player("P2", p2_stats, prior_strength=100, lam=0.90)

    N = 100_000
    print(f"Running {N:,} simulations...")

    p1_win_prob = run_simulation(p1, p2, N=N)

    print(f"{P1Name} win probability: {p1_win_prob:.4f}")
    print(f"{P2Name} win probability: {1 - p1_win_prob:.4f}")

Running 100,000 simulations...
Sinner win probability: 0.6289
Djokovic win probability: 0.3711


# Momentum Decay with Break Point Modelling

In [ ]:
import numpy as np

# ─────────────────────────────────────────
#  BETA MODEL
# ─────────────────────────────────────────

class BetaModel:
    """
    Tracks a single probability using a Beta distribution
    with exponential decay on in-match evidence.

    alpha_prior / beta_prior  : fixed career anchor, never touched after init
    alpha_match / beta_match  : in-match component, decayed each observation

    During warmup (n_obs < warmup threshold), returns pure prior.
    After warmup, blends prior + in-match.
    """

    def __init__(self, career_rate, prior_strength, lam=0.95, warmup=None):
        self.alpha_prior = career_rate * prior_strength
        self.beta_prior  = (1.0 - career_rate) * prior_strength

        self.alpha_match = 0.0
        self.beta_match  = 0.0

        self.lam    = lam
        self.warmup = warmup if warmup is not None else int(1 / (1 - lam))
        self.n_obs  = 0

    def get_p(self):
        if self.n_obs < self.warmup:
            return self.alpha_prior / (self.alpha_prior + self.beta_prior)
        total = (self.alpha_prior + self.beta_prior +
                 self.alpha_match + self.beta_match)
        return (self.alpha_prior + self.alpha_match) / total

    def update(self, won):
        self.alpha_match *= self.lam
        self.beta_match  *= self.lam
        if won:
            self.alpha_match += 1.0
        else:
            self.beta_match  += 1.0
        self.n_obs += 1

    def reset(self):
        self.alpha_match = 0.0
        self.beta_match  = 0.0
        self.n_obs       = 0


# ─────────────────────────────────────────
#  PLAYER
# ─────────────────────────────────────────

class Player:
    """
    Holds all Beta models for one player.

    Stats expected:
      first_in              : 1st serve in %
      win_first             : win % on 1st serve points (serving)
      win_second            : win % on 2nd serve points (serving)
      return_first          : win % returning opponent's 1st serve
      return_second         : win % returning opponent's 2nd serve
      bp_save_rate          : break point saved %
      bp_save_faced         : career break points faced (used as prior strength)
      bp_convert_rate       : break point converted %
      bp_convert_opps       : career break point opportunities (prior strength)
    """

    def __init__(self, name, stats, prior_strength=200, lam=0.95):
        self.name  = name
        self.stats = stats

        # Serve models — prior strength is tuned hyperparameter
        self.serve_first_model  = BetaModel(stats['win_first'],
                                            prior_strength, lam)
        self.serve_second_model = BetaModel(stats['win_second'],
                                            prior_strength, lam)

        # Return models
        self.return_first_model  = BetaModel(stats['return_first'],
                                             prior_strength, lam)
        self.return_second_model = BetaModel(stats['return_second'],
                                             prior_strength, lam)

        # Break point models — prior strength = actual observed count
        # warmup = 20 break point situations (rare, so kept at 20)
        self.bp_save_model = BetaModel(
            career_rate    = stats['bp_save_rate'],
            prior_strength = stats['bp_save_faced'],
            lam            = lam,
            warmup         = 20
        )
        self.bp_convert_model = BetaModel(
            career_rate    = stats['bp_convert_rate'],
            prior_strength = stats['bp_convert_opps'],
            lam            = lam,
            warmup         = 20
        )

    def reset(self):
        self.serve_first_model.reset()
        self.serve_second_model.reset()
        self.return_first_model.reset()
        self.return_second_model.reset()
        self.bp_save_model.reset()
        self.bp_convert_model.reset()


# ─────────────────────────────────────────
#  BREAK POINT DETECTION
# ─────────────────────────────────────────

def is_break_point(server_pts, receiver_pts):
    """
    Returns True if the current score is a break point situation
    (receiver is one point from winning the game).

    Score encoding: 0=0, 1=15, 2=30, 3=40, 4+=deuce/adv logic
    We track raw point counts and handle deuce separately.
    """
    # Receiver needs 4+ points and leads by 1 or more from deuce
    # Break point: receiver at 40 (3pts) and server < 40, OR receiver has adv
    if receiver_pts == 3 and server_pts < 3:
        return True
    # Advantage receiver (both reached deuce, receiver leads)
    if receiver_pts >= 4 and server_pts >= 3 and receiver_pts == server_pts + 1:
        return True
    return False


# ─────────────────────────────────────────
#  POINT SIMULATION
# ─────────────────────────────────────────

def sim_point(server: Player, receiver: Player,
              server_pts: int, receiver_pts: int):
    """
    Simulate one point.
    Uses break point models when score is a break point situation.
    Updates all relevant Beta models after outcome.
    Returns True if server wins.
    """
    bp = is_break_point(server_pts, receiver_pts)
    first_in = server.stats['first_in']

    if bp:
        # Break point: blend save % vs convert %
        p_save    = server.bp_save_model.get_p()
        p_convert = receiver.bp_convert_model.get_p()
        p_win = (p_save + (1.0 - p_convert)) / 2.0

        # Simulate outcome (no serve-type split on break points)
        server_won = np.random.random() < p_win

        # Update break point models for both players
        server.bp_save_model.update(server_won)
        receiver.bp_convert_model.update(not server_won)

        # Also update serve/return models — break points are still
        # serve points and count toward those career tendencies
        if np.random.random() < first_in:
            server.serve_first_model.update(server_won)
            receiver.return_first_model.update(not server_won)
        else:
            server.serve_second_model.update(server_won)
            receiver.return_second_model.update(not server_won)

    else:
        # Normal point: blend serve % vs return %
        p_serve_1st  = server.serve_first_model.get_p()
        p_serve_2nd  = server.serve_second_model.get_p()
        p_return_1st = receiver.return_first_model.get_p()
        p_return_2nd = receiver.return_second_model.get_p()

        p_win_1st = (p_serve_1st + (1.0 - p_return_1st)) / 2.0
        p_win_2nd = (p_serve_2nd + (1.0 - p_return_2nd)) / 2.0

        if np.random.random() < first_in:
            server_won = np.random.random() < p_win_1st
            server.serve_first_model.update(server_won)
            receiver.return_first_model.update(not server_won)
        else:
            server_won = np.random.random() < p_win_2nd
            server.serve_second_model.update(server_won)
            receiver.return_second_model.update(not server_won)

    return server_won


# ─────────────────────────────────────────
#  GAME SIMULATION
# ─────────────────────────────────────────

def sim_game(server: Player, receiver: Player):
    """
    Simulate one game. Returns True if server wins.
    Tracks raw point counts to detect break point situations.
    """
    # Raw point counts (not tennis notation)
    score = [0, 0]   # [server_pts, receiver_pts]

    while True:
        server_won = sim_point(server, receiver,
                               server_pts   = score[0],
                               receiver_pts = score[1])

        if server_won:
            score[0] += 1
        else:
            score[1] += 1

        # Win: 4+ points, lead by 2
        if score[0] >= 4 and score[0] - score[1] >= 2:
            return True
        if score[1] >= 4 and score[1] - score[0] >= 2:
            return False


# ─────────────────────────────────────────
#  TIEBREAK SIMULATION
# ─────────────────────────────────────────

def sim_tiebreak(p1: Player, p2: Player, p1_serves_first: bool):
    """
    Simulate a tiebreak. First to 7, win by 2.
    No break point logic in tiebreaks (no break points exist).
    Server alternates: 1 point then every 2.
    """
    score = [0, 0]
    point_count = 0

    while True:
        if point_count == 0:
            p1_serves = p1_serves_first
        else:
            p1_serves = p1_serves_first == (point_count % 2 == 0)

        server   = p1 if p1_serves else p2
        receiver = p2 if p1_serves else p1

        # Tiebreak points are never break points — pass impossible score
        server_won = sim_point(server, receiver,
                               server_pts=0, receiver_pts=0)
        p1_won = server_won if p1_serves else not server_won

        if p1_won:
            score[0] += 1
        else:
            score[1] += 1

        point_count += 1

        if score[0] >= 7 and score[0] - score[1] >= 2:
            return True
        if score[1] >= 7 and score[1] - score[0] >= 2:
            return False


# ─────────────────────────────────────────
#  SET SIMULATION
# ─────────────────────────────────────────

def sim_set(p1: Player, p2: Player, p1_serves_first: bool):
    """
    Simulate one set. Returns (True if p1 wins, p1_serving next set).
    """
    games = [0, 0]
    p1_serving = p1_serves_first

    while True:
        server   = p1 if p1_serving else p2
        receiver = p2 if p1_serving else p1

        server_won  = sim_game(server, receiver)
        p1_won_game = server_won if p1_serving else not server_won

        if p1_won_game:
            games[0] += 1
        else:
            games[1] += 1

        p1_serving = not p1_serving

        # Tiebreak at 6-6
        if games[0] == 6 and games[1] == 6:
            p1_won_tb = sim_tiebreak(p1, p2, p1_serving)
            if p1_won_tb:
                games[0] += 1
            else:
                games[1] += 1
            return (games[0] > games[1]), p1_serving

        if games[0] >= 6 and games[0] - games[1] >= 2:
            return True, p1_serving
        if games[1] >= 6 and games[1] - games[0] >= 2:
            return False, p1_serving


# ─────────────────────────────────────────
#  MATCH SIMULATION
# ─────────────────────────────────────────

def sim_match(p1: Player, p2: Player,
              p1_serves_first: bool = True,
              best_of: int = 3):
    """
    Simulate one full match. Resets both players at start.
    Returns True if p1 wins.
    """
    p1.reset()
    p2.reset()

    sets_needed = best_of // 2 + 1
    sets = [0, 0]
    p1_serving = p1_serves_first

    while sets[0] < sets_needed and sets[1] < sets_needed:
        p1_won_set, p1_serving = sim_set(p1, p2, p1_serving)
        if p1_won_set:
            sets[0] += 1
        else:
            sets[1] += 1

    return sets[0] > sets[1]


# ─────────────────────────────────────────
#  RUN SIMULATION
# ─────────────────────────────────────────

def run_simulation(p1: Player, p2: Player,
                   N: int = 100_000,
                   p1_serves_first: bool = True,
                   best_of: int = 3):
    wins = 0
    for _ in range(N):
        if sim_match(p1, p2, p1_serves_first, best_of):
            wins += 1

    p1_prob = wins / N
    p2_prob = 1 - p1_prob

    p1_se = np.sqrt(p1_prob * p2_prob / N)
    p2_se = np.sqrt(p2_prob * p1_prob / N)  # same value, symmetric

    return p1_prob, p1_se, p2_prob, p2_se


# ─────────────────────────────────────────
#  EXAMPLE USAGE
# ─────────────────────────────────────────

if __name__ == "__main__":

    P1Name = "Sinner"
    P2Name = "Djokovic"
    p1_stats= {
        'first_in':      0.62,   
        'win_first':     0.81,
        'win_second':    0.59,
        'return_first':  0.33,
        'return_second': 0.59,
        'bp_save_rate':     0.77,
        'bp_save_faced':    62,   # actual career count → prior strength
        'bp_convert_rate':  0.38,
        'bp_convert_opps':  172,   # actual career count → prior strength
    }

    p2_stats = {
        'first_in':      0.66,
        'win_first':     0.77,
        'win_second':    0.57,
        'return_first':  0.31,   
        'return_second': 0.51,   
        'bp_save_rate':     0.70,
        'bp_save_faced':    73,   # actual career count → prior strength
        'bp_convert_rate':  0.39,
        'bp_convert_opps':  85,   # actual career count → prior strength
    }


    p1 = Player("P1", p1_stats, prior_strength=50, lam=0.90)
    p2 = Player("P2", p2_stats, prior_strength=50, lam=0.90)

    N = 50_000
    print(f"Running {N:,} simulations...")
    p1_win_prob, p1_se, p2_win_prob, p2_se = run_simulation(p1, p2, N=N)
    print(f"{P1Name} win probability : {p1_win_prob:.4f} ± {p1_se:.4f}")
    print(f"P1 95% CI : ({p1_win_prob - 1.96*p1_se:.4f}, {p1_win_prob + 1.96*p1_se:.4f})")
    print("---------------------------------")
    print(f"{P2Name} win probability : {p2_win_prob:.4f} ± {p2_se:.4f}")
    print(f"P2 95% CI : ({p2_win_prob - 1.96*p2_se:.4f}, {p2_win_prob + 1.96*p2_se:.4f})")

Running 50,000 simulations...
Sinner win probability : 0.6285 ± 0.0022
P1 95% CI : (0.6242, 0.6327)
---------------------------------
Djokovic win probability : 0.3715 ± 0.0022
P2 95% CI : (0.3673, 0.3758)
